In [2]:
import os
import time
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.densenet import DenseNet201, preprocess_input

# 1. Setup paths
DATA_DIR = "../data"
IMAGES_DIR = os.path.join(DATA_DIR, "Images")
FEATURES_PATH = os.path.join(DATA_DIR, "features.pkl")

# 2. Configure Native CUDA GPU (NVIDIA RTX 3060)
gpus = tf.config.list_physical_devices('GPU')
print(f"[GPU] Devices detected: {len(gpus)}")
for i, gpu in enumerate(gpus):
    print(f"  -> GPU {i}: {gpu.name}")

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf.config.set_visible_devices(gpus[0], 'GPU')
        print(f"[GPU] Active compute target set to: {gpus[0].name}")
    except RuntimeError as e:
        print(f"[GPU] Configuration note: {e}")

# 3. Load DenseNet201 pretrained on ImageNet (Global Average Pooling)
# include_top=False + pooling='avg' directly produces (None, 1920) feature vectors
model = DenseNet201(weights="imagenet", include_top=False, pooling="avg")
print(f"Model loaded successfully! Output feature vector shape: {model.output_shape}")

[GPU] Devices detected: 0
Model loaded successfully! Output feature vector shape: (None, 1920)


In [3]:
import gc
import os
import pickle
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# 1. Cap TensorFlow memory footprint
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

gpus = tf.config.list_physical_devices("GPU")
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
  except RuntimeError as e:
    pass

# 2. Paths & checkpoint setup
FEATURES_PATH = "../data/features.pkl"
os.makedirs("../data", exist_ok=True)

if os.path.exists(FEATURES_PATH):
  with open(FEATURES_PATH, "rb") as f:
    features = pickle.load(f)
  print(f"Resuming: {len(features)} images already extracted.")
else:
  features = {}

all_images = sorted([
    f
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])
remaining_images = [img for img in all_images if img not in features]
total_images = len(all_images)
print(f"Total: {total_images} | Remaining to extract: {len(remaining_images)}")

# 3. Micro-batch settings tuned for 5.6 GB RAM ceiling
BATCH_SIZE = 8
SAVE_INTERVAL = 200

batch_imgs = []
batch_names = []
start_time = time.time()

for idx, img_name in enumerate(remaining_images):
  img_path = os.path.join(IMAGES_DIR, img_name)
  try:
    with load_img(img_path, target_size=(224, 224)) as img:
      arr = img_to_array(img, dtype="float32")
      arr = preprocess_input(arr)
      batch_imgs.append(arr)
      batch_names.append(img_name)
  except Exception:
    continue

  # Predict when batch fills up or on the last item
  if len(batch_imgs) == BATCH_SIZE or idx == len(remaining_images) - 1:
    batch_tensor = np.array(batch_imgs, dtype=np.float32)
    # Extract directly using the model loaded earlier
    preds = model.predict(batch_tensor, verbose=0)

    for name, pred in zip(batch_names, preds):
      features[name] = pred.flatten().astype(np.float32)

    batch_imgs.clear()
    batch_names.clear()

  # Periodic save to disk & explicit RAM release
  if (idx + 1) % SAVE_INTERVAL == 0 or idx == len(remaining_images) - 1:
    with open(FEATURES_PATH, "wb") as f:
      pickle.dump(features, f, protocol=pickle.HIGHEST_PROTOCOL)
    gc.collect()
    done = len(features)
    elapsed = time.time() - start_time
    print(
        f"Saved checkpoint: [{done}/{total_images}] images extracted"
        f" ({done/total_images*100:.1f}%)"
    )

print(
    f"\nExtraction complete! Saved all {len(features)} features to"
    f" {FEATURES_PATH}"
)

Resuming: 8091 images already extracted.
Total: 8091 | Remaining to extract: 0

Extraction complete! Saved all 8091 features to ../data/features.pkl


In [5]:
import os
import pickle

DATA_DIR = "../data"
FEATURES_PATH = os.path.join(DATA_DIR, "features.pkl")

# If features isn't already loaded in memory, load it from the saved file
if "features" not in globals() or len(features) == 0:
  with open(FEATURES_PATH, "rb") as f:
    features = pickle.load(f)

file_size_mb = os.path.getsize(FEATURES_PATH) / (1024 * 1024)
print(f"Features file location: {FEATURES_PATH}")
print(f"Total features stored: {len(features)}")
print(f"Feature file size: {file_size_mb:.2f} MB")

# Sanity check directly using keys from features
sample_key = next(iter(features.keys()))
print(f"Sample key: {sample_key}")
print(f"Sample feature vector shape: {features[sample_key].shape}")

Features file location: ../data/features.pkl
Total features stored: 8091
Feature file size: 59.74 MB
Sample key: 1000268201_693b08cb0e.jpg
Sample feature vector shape: (1920,)
